[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/03_band_atlas.ipynb)

# 3 · The band atlas

*Brown Dwarf Spectra On-Ramp · about 35 minutes*

**By the end of this notebook you will be able to:**
- name the absorbers that shape a brown dwarf spectrum between 0.75 and 5 µm, and where they act
- read 'molecule bar' plots
- explain two traps: overlapping bands, and band wings reaching into windows
- explain why band depth tracks absorption, not abundance

> **First time here?** Do *File → Save a copy in Drive* before you change anything (notebook 0 explains why).
> Then run the **Setup** cell below. Run cells in order, top to bottom, with **Shift + Enter**.

In [ ]:
# ▶ SETUP - run this cell first. It is safe to run more than once.
REPO_URL = "https://github.com/zshakespear/bd-spectra-onramp.git"
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:                       # running in Google Colab
    ROOT = Path("/content/bd-spectra-onramp")
    if not (ROOT / "bdtools").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "--quiet"], check=False)
else:                                                   # running in Jupyter on your own computer
    ROOT = Path.cwd()
    while not (ROOT / "bdtools").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import bdtools as bd
bd.setup_notebook()

## 1. The atlas

The band positions live in one instructor-editable file, `data/bands.yaml`, so every plot and the
field guide use the same ranges and colours. Each row says where its numbers come from.
`status` tells you where the row comes from: **spec** = stated in Rustamkulov et al. (2026, Sec. 3);
**checked** = range from the band table of Cushing et al. (2005); **uncertain** = listed by Cushing et al. (2005),
but whether SPHEREx can separate it at all is unclear.

In [ ]:
import pandas as pd
rows = []
for name, ab in bd.load_bands()["absorbers"].items():
    for b in ab["bands"]:
        rows.append({"absorber": name, "range (um)": f"{b['range_um'][0]:.2f}-{b['range_um'][1]:.2f}",
                     "types": b["types"], "status": b["status"], "overlaps": b.get("overlaps", ""),
                     "source": b["source"]})
pd.set_option("display.max_colwidth", 70)
pd.DataFrame(rows)

**NH<sub>3</sub>** is in the list for completeness: it is unlikely to be detectable with SPHEREx
(Rustamkulov et al. 2026), so don't go looking for it. Uncertain rows are hidden from plots unless you ask.

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
SPECTRAL_TYPE = "L8"
SHOW_UNCERTAIN = False

In [ ]:
spec = bd.load_reference(SPECTRAL_TYPE)
fig, ax = bd.plot_spectrum(spec, log=True, include_uncertain=SHOW_UNCERTAIN,
                           title=f"{spec.name} with molecule bars")

## 2. Footprints: remove one absorber at a time

The cleanest way to see what an absorber does is to compute the spectrum with and without it.
The curated model grids don't provide spectra with single absorbers removed, so we use the toy —
at SPHEREx-like resolution, so the footprints look the way SPHEREx would see them.

> 🧸 **TOY MODEL** — real band shapes and strengths differ; use this for *where* and *how*, not *how much*.

In [ ]:
r1 = bd.load_toy("remove_one")
labels = list(r1.labels)
full = r1.spherex_flux[labels.index("all")]
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7.5), gridspec_kw={"height_ratios": [2, 1.4]})
axes[0].plot(r1.spherex_wave_um, full, "k-o", ms=2, label="all toy absorbers")
for lab, f in zip(labels, r1.spherex_flux):
    if lab == "all":
        continue
    name = lab[3:]
    c = bd.molecule_color(name)
    axes[0].plot(r1.spherex_wave_um, f, color=c, lw=1.2, label=lab)
    axes[1].plot(r1.spherex_wave_um, f / full, color=c, lw=1.5, label=lab)
axes[0].set_yscale("log"); axes[0].legend(ncol=3, fontsize=8); axes[0].set_ylabel(r"$F_\lambda$ (toy)")
axes[1].axhline(1, color="0.4"); axes[1].set_yscale("log")
axes[1].set_ylabel("without X / with all"); axes[1].set_xlabel("Wavelength [µm]")
axes[0].set_title("TOY: footprint of each absorber", fontsize=10);

Each curve in the bottom panel is one absorber's **footprint**: where the spectrum would get brighter
if that absorber disappeared.

## 3. Trap 1 — overlapping bands

Look at 1.1–1.2 µm and 2.2–2.5 µm in the footprint plot: more than one absorber's footprint is above 1 there.
When bands overlap, a deeper trough doesn't tell you *which* absorber changed. In real spectra:

* 1.1–1.2 µm: H<sub>2</sub>O and (in T dwarfs) CH<sub>4</sub>;
* 2.2–2.5 µm: CH<sub>4</sub>, CO, and H<sub>2</sub> collision-induced absorption (CIA);
* 3–3.6 µm: CH<sub>4</sub> on the wing of the broad 2.7 µm H<sub>2</sub>O band.

Always check the atlas for *every* absorber in a region before naming one.

## 4. Trap 2 — band wings reach into the windows

The footprints are not zero in the windows. Removing H<sub>2</sub>O changes the flux at the J and H peaks too, because
the wings of its bands reach into the windows. **The height of a window peak depends on its neighbours.**

In [ ]:
for target in [1.27, 1.60, 4.00]:
    i = np.argmin(abs(r1.spherex_wave_um - target))
    changes = {lab: f"{(f[i] / full[i] - 1) * 100:+.0f}%" for lab, f in zip(labels, r1.spherex_flux) if lab != "all"}
    print(f"window at {target} um, flux change when removing:", changes)

## 5. Trap 3 — depth tracks absorption, not abundance

From notebook 2: band depth grows roughly with the *log* of the absorber amount and saturates. So when two
spectra differ in a band, say "**more absorption** at 3.3 µm", not "more methane". A temperature difference
also changes many bands at once (next cell): a **global** change, while removing one absorber changes only
its own bands — a **local** change.

In [ ]:
tv = bd.load_toy("temp_vs_absorber")
base = tv.spherex_flux[0]
fig, ax = plt.subplots()
bd.shade_bands(ax)
for lab, f, c in zip(tv.labels[1:], tv.spherex_flux[1:], bd.LINE_COLORS[1:]):
    ax.plot(tv.spherex_wave_um, f / base, "o-", ms=2, color=c, label=f"{lab}  /  {tv.labels[0]}")
ax.axhline(1, color="0.4"); ax.set_yscale("log"); ax.legend(fontsize=8)
ax.set_xlabel("Wavelength [µm]"); ax.set_ylabel("ratio"); ax.set_title("TOY: global vs local change", fontsize=10);

## 6. Where the real band data come from (pointers only)

Real models use **line lists** — laboratory and theoretical catalogues of millions to billions of individual
molecular transitions (e.g. the ExoMol and HITRAN/HITEMP databases) — and **radiative-transfer codes** that
compute spectra from them (e.g. PICASO, petitRADTRANS). You will not need them in this course; the model
grids in notebook 5 were built with tools like these.

### Exercise 3.1
In the footprint plot, which toy absorber affects the most wavelengths? Is that also true in the real atlas table?

<details><summary><b>Answer</b> (try first, then click)</summary>

H<sub>2</sub>O: it has bands throughout the near-infrared (0.93, 1.15, 1.4, 1.9, 2.7 µm). In real L, T and Y dwarfs,
water is the dominant absorber, carving the troughs between the Y, J, H, K and 4 µm peaks.

</details>

### Exercise 3.2
Two spectra differ only at 2.2–2.5 µm. List every absorber that could be responsible.

<details><summary><b>Answer</b> (try first, then click)</summary>

CH<sub>4</sub>, CO (2.3 µm overtone), and H<sub>2</sub> CIA — and possibly the wing of the 1.9 µm H<sub>2</sub>O band at the
short end. Gravity and metallicity change CIA, so they belong on the list of explanations too (notebook 5).

</details>

## Summary

- H<sub>2</sub>O shapes the whole near-infrared; CH<sub>4</sub> appears at 3–3.6 µm after L4 and defines T dwarfs; CO and CO<sub>2</sub> make notches beyond 4.15 µm; H<sub>2</sub> CIA is broad, ~1.7–3 µm.
- Footprints (spectrum without X / with X) show where an absorber acts.
- Bands overlap, and band wings set the height of window peaks.
- Temperature changes many bands at once; one absorber changes only its own bands.
- Field guide: [Band atlas](../field_guide/band_atlas.md)

**Field guide:** the pages linked above are the short versions to keep open while you work.

## Self-check

Answer these without scrolling up. If one is hard, revisit that section.

1. Name the absorbers that could deepen the trough near 1.15 µm in a T dwarf.
<details><summary><b>Answer</b> (try first, then click)</summary>

H<sub>2</sub>O and CH<sub>4</sub>.

</details>

2. Why can removing H<sub>2</sub>O change the J-peak flux?
<details><summary><b>Answer</b> (try first, then click)</summary>

The wings of neighbouring H<sub>2</sub>O bands reach into the J window.

</details>

3. How does a global (temperature) change look different from a local (one absorber) change in a ratio plot?
<details><summary><b>Answer</b> (try first, then click)</summary>

A temperature change moves many bands and the overall slope at once; an absorber change is confined to that absorber's bands.

</details>

4. Why is NH<sub>3</sub> in the atlas but not on the plots?
<details><summary><b>Answer</b> (try first, then click)</summary>

It is unlikely to be detectable with SPHEREx (Rustamkulov et al. 2026).

</details>

**Next:** [Notebook 4 · The spectral sequence](04_spectral_sequence.ipynb)